# F3 - Algoritmos, recursividad y mediciones de complejidad

## Proyecto
**Analisis global de la industria fitness y gimnasios 2000-2026**

## Integrantes
- Wilson Arevalo
- Luis Espinosa
- Mauricio Ortega
- Eduardo Garrido

## Proposito
Esta fase se orienta al diseno e implementacion de algoritmos estructurados y recursivos en Python, utilizando el dataset procesado generado en Fase 2. El foco no esta en limpiar datos, sino en construir algoritmos, medir complejidad temporal y espacial, y documentar oportunidades de optimizacion.

## Relacion con Fase 2

La Fase 2 genero el dataset procesado `data/processed/gym_data_processed.csv`. En Fase 3 se utiliza este archivo como entrada para implementar algoritmos sobre datos ya preparados. Por eso el preprocesamiento aqui es minimo.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Configuracion de rutas e importacion de scripts

El notebook esta dentro de `notebooks/`, por eso `Path("..").resolve()` apunta a la raiz del repositorio. Desde ahi se importan los scripts de `src/fase3/`.

In [2]:
PROJECT_ROOT = Path("..").resolve()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "gym_data_processed.csv"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Raiz del proyecto:", PROJECT_ROOT)
print("Ruta dataset procesado:", DATA_PATH)

from src.fase3.algoritmos import (
    calcular_ranking_paises,
    maximo_iterativo,
    maximo_recursivo,
    maximo_recursivo_optimizado,
    promedio_manual_por_region,
    promedio_pandas_por_region,
    busqueda_lineal_pais,
    filtrar_por_periodo,
)
from src.fase3.mediciones import (
    medir_tiempo,
    medir_memoria,
    medir_tiempo_y_memoria,
    construir_resumen_medicion,
)
from src.fase3.analizador_fitness import AnalizadorFitness

Raiz del proyecto: C:\Users\warevalo\Documents\GitHub\gym-fitness-analytics
Ruta dataset procesado: C:\Users\warevalo\Documents\GitHub\gym-fitness-analytics\data\processed\gym_data_processed.csv


ModuleNotFoundError: No module named 'src.fase3.algoritmos'

## Diagnostico de estructura

Esta celda confirma que los scripts y el dataset estan en las rutas esperadas.

In [ ]:
print("Existe src:", (PROJECT_ROOT / "src").exists())
print("Existe src/fase3:", (PROJECT_ROOT / "src" / "fase3").exists())
print("Existe algoritmos.py:", (PROJECT_ROOT / "src" / "fase3" / "algoritmos.py").exists())
print("Existe mediciones.py:", (PROJECT_ROOT / "src" / "fase3" / "mediciones.py").exists())
print("Existe analizador_fitness.py:", (PROJECT_ROOT / "src" / "fase3" / "analizador_fitness.py").exists())
print("Existe dataset procesado:", DATA_PATH.exists())

## Carga del dataset procesado

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"No se encontro el dataset procesado en la ruta: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
print("Dataset procesado cargado correctamente.")
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
df.head()

## Validacion basica de columnas para F3

In [ ]:
columnas_necesarias = [
    "country", "region", "year", "gym_penetration_rate",
    "memberships_per_100k", "gyms_per_100k",
    "revenue_per_membership_usd", "periodo",
]
faltantes = [col for col in columnas_necesarias if col not in df.columns]
if faltantes:
    raise ValueError(f"Faltan columnas necesarias para F3: {faltantes}")
print("Todas las columnas necesarias para F3 estan presentes.")

## Preprocesamiento minimo para Fase 3

Solo se seleccionan columnas relevantes y se eliminan nulos puntuales en variables usadas por los algoritmos.

In [ ]:
columnas_f3 = [
    "country", "region", "year", "gym_penetration_rate",
    "memberships_per_100k", "gyms_per_100k",
    "revenue_per_membership_usd", "periodo",
]
df_f3 = df[columnas_f3].copy()
df_f3 = df_f3.dropna(subset=["gym_penetration_rate", "memberships_per_100k", "gyms_per_100k"])
print("Dataset preparado para F3.")
print(f"Filas disponibles: {df_f3.shape[0]}")
print(f"Columnas disponibles: {df_f3.shape[1]}")
df_f3.head()

## Algoritmo 1: Ranking iterativo de paises

Enfoque estructurado. Recorre el DataFrame, filtra valores validos, crea una lista auxiliar y ordena.

- Complejidad temporal: `O(n log n)`.
- Complejidad espacial: `O(n)`.

In [ ]:
ranking_memberships = calcular_ranking_paises(df_f3, columna="memberships_per_100k", top_n=10)
pd.DataFrame(ranking_memberships)

## Algoritmo 2: Maximo iterativo y recursivo

Se comparan tres implementaciones: iterativa, recursiva con slicing y recursiva optimizada con indices.

In [ ]:
valores_memberships = df_f3["memberships_per_100k"].dropna().astype(float).tolist()
print(f"Cantidad de valores disponibles: {len(valores_memberships)}")
print(f"Primeros 5 valores: {valores_memberships[:5]}")

In [ ]:
max_iterativo = maximo_iterativo(valores_memberships)
max_recursivo = maximo_recursivo(valores_memberships)
max_recursivo_opt = maximo_recursivo_optimizado(valores_memberships)
print("Maximo iterativo:", max_iterativo)
print("Maximo recursivo:", max_recursivo)
print("Maximo recursivo optimizado:", max_recursivo_opt)
assert max_iterativo == max_recursivo == max_recursivo_opt
print("Las tres implementaciones retornan el mismo resultado.")

## Algoritmo 3: Promedio manual por region vs Pandas

Comparacion entre una implementacion manual con ciclos y una version con `groupby()` de Pandas.

In [ ]:
promedio_manual = promedio_manual_por_region(df_f3, "gym_penetration_rate")
promedio_pandas = promedio_pandas_por_region(df_f3, "gym_penetration_rate")
print("Promedio manual por region:")
display(pd.Series(promedio_manual).sort_values(ascending=False))
print("Promedio Pandas por region:")
display(promedio_pandas)

## Uso exploratorio de POO

Se incorpora una clase simple `AnalizadorFitness` como aproximacion inicial a POO. No se implementa arquitectura avanzada.

In [ ]:
analizador = AnalizadorFitness(df_f3)
analizador.resumen_basico()

In [ ]:
analizador.obtener_top_paises(columna="memberships_per_100k", top_n=10)

## Medicion de complejidad temporal y espacial

Se miden tiempos promedio y memoria pico con distintos tamanos de muestra.

In [ ]:
tamanios_base = [100, 500, 1000, 5000, len(df_f3)]
tamanios_muestra = []
for n in tamanios_base:
    if n <= len(df_f3) and n not in tamanios_muestra:
        tamanios_muestra.append(n)
print("Tamanios de muestra utilizados:", tamanios_muestra)

In [ ]:
resultados_mediciones = []

for n in tamanios_muestra:
    df_muestra = df_f3.head(n).copy()
    valores_muestra = df_muestra["memberships_per_100k"].dropna().astype(float).tolist()

    medicion_ranking = medir_tiempo_y_memoria(calcular_ranking_paises, df_muestra, "memberships_per_100k", 10, repeticiones=5)
    resultados_mediciones.append(construir_resumen_medicion("Ranking iterativo", "Estructurado", n, medicion_ranking, "O(n log n)", "O(n)"))

    medicion_max_iterativo = medir_tiempo_y_memoria(maximo_iterativo, valores_muestra, repeticiones=5)
    resultados_mediciones.append(construir_resumen_medicion("Maximo iterativo", "Estructurado", n, medicion_max_iterativo, "O(n)", "O(1)"))

    medicion_max_recursivo = medir_tiempo_y_memoria(maximo_recursivo, valores_muestra, repeticiones=5)
    resultados_mediciones.append(construir_resumen_medicion("Maximo recursivo con slicing", "Recursivo", n, medicion_max_recursivo, "O(n)", "O(n)"))

    medicion_max_recursivo_opt = medir_tiempo_y_memoria(maximo_recursivo_optimizado, valores_muestra, repeticiones=5)
    resultados_mediciones.append(construir_resumen_medicion("Maximo recursivo optimizado", "Recursivo optimizado", n, medicion_max_recursivo_opt, "O(n)", "O(log n)"))

    medicion_promedio_manual = medir_tiempo_y_memoria(promedio_manual_por_region, df_muestra, "gym_penetration_rate", repeticiones=5)
    resultados_mediciones.append(construir_resumen_medicion("Promedio manual por region", "Estructurado", n, medicion_promedio_manual, "O(n)", "O(k)"))

    medicion_promedio_pandas = medir_tiempo_y_memoria(promedio_pandas_por_region, df_muestra, "gym_penetration_rate", repeticiones=5)
    resultados_mediciones.append(construir_resumen_medicion("Promedio Pandas por region", "Pandas groupby", n, medicion_promedio_pandas, "O(n) aprox.", "O(k)"))

df_mediciones = pd.DataFrame(resultados_mediciones)
df_mediciones

## Visualizacion de tiempos de ejecucion

In [ ]:
plt.figure(figsize=(12, 7))
for algoritmo in df_mediciones["algoritmo"].unique():
    datos_algoritmo = df_mediciones[df_mediciones["algoritmo"] == algoritmo]
    plt.plot(datos_algoritmo["n"], datos_algoritmo["tiempo_promedio_seg"], marker="o", label=algoritmo)
plt.xlabel("Tamano de muestra (n)")
plt.ylabel("Tiempo promedio de ejecucion (segundos)")
plt.title("Comparacion de tiempo promedio por algoritmo")
plt.legend()
plt.grid(True)
plt.show()

## Visualizacion de memoria pico

In [ ]:
plt.figure(figsize=(12, 7))
for algoritmo in df_mediciones["algoritmo"].unique():
    datos_algoritmo = df_mediciones[df_mediciones["algoritmo"] == algoritmo]
    plt.plot(datos_algoritmo["n"], datos_algoritmo["memoria_pico_kb"], marker="o", label=algoritmo)
plt.xlabel("Tamano de muestra (n)")
plt.ylabel("Memoria pico (KB)")
plt.title("Comparacion de memoria pico por algoritmo")
plt.legend()
plt.grid(True)
plt.show()

## Interpretacion de resultados

El ranking iterativo tiene complejidad `O(n log n)` porque el ordenamiento domina el costo. El maximo iterativo tiene complejidad `O(n)` y memoria `O(1)`. El maximo recursivo con slicing tambien es `O(n)`, pero consume mas memoria porque crea sublistas. La version recursiva optimizada reduce ese problema usando indices.

La comparacion entre promedio manual y Pandas muestra que la implementacion manual ayuda a comprender el flujo algoritmico, mientras que Pandas entrega mayor legibilidad y usa operaciones optimizadas.

## Relacion con el foro tecnico de Fase 3

El foro tecnico permitio discutir cuando conviene usar programacion estructurada y cuando usar recursividad. En este notebook se aplican ambos enfoques para comparar ventajas, limitaciones y oportunidades de optimizacion.

## Conclusiones

La Fase 3 permitio aplicar programacion estructurada y recursiva sobre el dataset procesado en Fase 2. Se implementaron algoritmos de ranking, busqueda de maximos, promedios por region y mediciones de tiempo y memoria.

Como mejora futura, se recomienda ampliar mediciones, evitar slicing en funciones recursivas cuando el dataset crezca y mantener la logica reutilizable dentro de `src/`.

## Referencias

Cormen, T. H., Leiserson, C. E., Rivest, R. L., & Stein, C. (2022). *Introduction to algorithms* (4th ed.). MIT Press.

Python Software Foundation. (s. f.). *Python documentation*. https://docs.python.org/3/

The pandas development team. (s. f.). *Pandas documentation*. https://pandas.pydata.org/docs/

Project Jupyter. (s. f.). *Jupyter documentation*. https://docs.jupyter.org/

National Academies of Sciences, Engineering, and Medicine. (2019). *Reproducibility and replicability in science*. The National Academies Press. https://doi.org/10.17226/25303